In [98]:
import os
import subprocess
import tempfile

import numpy as np
import matplotlib.pyplot as plt

In [99]:
class PoseForceViewer:
    def __init__(self, keypoints, forces):
        self.kpts = keypoints
        self.forces = forces
        self.frame = 0
        self.num_frames = keypoints.shape[0]

        # COCO Connections
        self.skeleton = [
            (5,7),(7,9),
            (6,8),(8,10),
            (5,6),
            (5,11),(6,12),
            (11,12),
            (11,13),(13,15),
            (12,14),(14,16)
        ]

        self.fig, self.ax = plt.subplots()
        self.fig.canvas.mpl_connect('scroll_event', self.on_scroll)

        self.draw()

    def set_frame(self, frame):
        self.frame = max(0, min(frame, self.num_frames - 1))
        self.draw()

    def next_frame(self):
        if self.frame < self.num_frames - 1:
            self.frame += 1
            self.draw()

    def prev_frame(self):
        if self.frame > 0:
            self.frame -= 1
            self.draw()

    def draw(self):
        self.ax.clear()

        k = self.kpts[self.frame]
        f = self.forces[self.frame]

        self.ax.scatter(k[:,0], k[:,1], c='red')

        for i, j in self.skeleton:
            self.ax.plot([k[i,0], k[j,0]],
                         [k[i,1], k[j,1]], 'blue')

        if not np.any(np.isnan(k[15])): # 15 for left ankle
            x, y = k[15]
            fx, fy, fz = f[0], f[1], f[2]

            self.ax.arrow(x, y,
                          fx*scale, -fy*scale,
                          color='purple',
                          head_width=5)
            
        if not np.any(np.isnan(k[16])): # 16 for right ankle
            x, y = k[16]
            fx, fy, fz = f[3], f[4], f[5]

            self.ax.arrow(x, y,
                          fx*scale, -fy*scale,
                          color='purple',
                          head_width=5)
            
        x_min = np.nanmin(self.kpts[:,:,0]) - 20
        x_max = np.nanmax(self.kpts[:,:,0]) + 20
        y_min = np.nanmin(self.kpts[:,:,1]) - 20
        y_max = np.nanmax(self.kpts[:,:,1]) + 20

        self.ax.set_title(f"Frame {self.frame}")
        self.ax.set_xlim(x_min, x_max)
        self.ax.set_ylim(y_max, y_min)
        self.ax.set_aspect('equal')
        self.fig.canvas.draw_idle()
            
    def on_scroll(self, event):
        if event.button == 'up':
            self.frame = min(self.frame + 1, self.num_frames - 1)
        if event.button == 'down':
            self.frame = max(self.frame - 1, 0)

        self.draw()

In [100]:
def save_force_video(keypoints, forces, out_path='out_files/force_video.mp4', fps=30, scale=5, figsize=(6,6), dpi=100):
    keypoints = np.asarray(keypoints)
    forces = np.asarray(forces)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    x_min = np.nanmin(keypoints[:,:,0]) - 20
    x_max = np.nanmax(keypoints[:,:,0]) + 20
    y_min = np.nanmin(keypoints[:,:,1]) - 20
    y_max = np.nanmax(keypoints[:,:,1]) + 20

    skeleton = [
        (5,7),(7,9),
        (6,8),(8,10),
        (5,6),
        (5,11),(6,12),
        (11,12),
        (11,13),(13,15),
        (12,14),(14,16)
    ]

    with tempfile.TemporaryDirectory() as tmpdir:
        for frame_idx in range(keypoints.shape[0]):
            k = keypoints[frame_idx]
            f = forces[frame_idx]

            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
            ax.scatter(k[:,0], k[:,1], c='red')
            for i, j in skeleton:
                ax.plot([k[i,0], k[j,0]], [k[i,1], k[j,1]], 'blue')

            if not np.any(np.isnan(k[16])):
                x, y = k[16]
                fx, fy, fz = f[0], f[1], f[2]
                ax.arrow(x, y, fx * scale, -fy * scale, color='purple', head_width=5)

            if not np.any(np.isnan(k[15])):
                x, y = k[15]
                fx, fy, fz = f[3], f[4], f[5]
                ax.arrow(x, y, fx * scale, -fy * scale, color='purple', head_width=5)

            ax.set_title(f'Frame {frame_idx}')
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)
            ax.set_aspect('equal')

            filename = os.path.join(tmpdir, f'frame_{frame_idx:04d}.png')
            fig.savefig(filename, dpi=dpi)
            plt.close(fig)

        cmd = [
            'ffmpeg',
            '-y',
            '-framerate', str(fps),
            '-i', os.path.join(tmpdir, 'frame_%04d.png'),
            '-c:v', 'libx264',
            '-pix_fmt', 'yuv420p',
            out_path,
        ]
        print('Running ffmpeg...')
        subprocess.run(cmd, check=True)

    print(f'Video saved to {out_path}')

In [ ]:
kps = np.load("yolo_kps/keypoints_2.npy")

pred = np.load("yolo_kps/pred_force.npy")
forces = pred.squeeze(axis=1)

In [102]:
forces.min(), forces.max()

(np.float32(-0.29991114), np.float32(6.2614126))

In [103]:
output_video = 'out_files/force_single_leg_jump.mp4'
save_force_video(kps, forces, out_path=output_video, fps=30, scale=10)
print('Saved video to', output_video)

Running ffmpeg...


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Video saved to out_files/force_single_leg_jump.mp4
Saved video to out_files/force_single_leg_jump.mp4


[out#0/mp4 @ 0x635fb1fd4140] video:458kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 1.581093%
frame=  551 fps=0.0 q=-1.0 Lsize=     465kB time=00:00:18.26 bitrate= 208.7kbits/s speed=58.1x    
[libx264 @ 0x635fb1fd5640] frame I:3     Avg QP: 9.97  size:  9211
[libx264 @ 0x635fb1fd5640] frame P:146   Avg QP:23.87  size:  1448
[libx264 @ 0x635fb1fd5640] frame B:402   Avg QP:31.84  size:   571
[libx264 @ 0x635fb1fd5640] consecutive B-frames:  0.9%  3.6%  5.4% 90.0%
[libx264 @ 0x635fb1fd5640] mb I  I16..4: 78.1% 11.8% 10.1%
[libx264 @ 0x635fb1fd5640] mb P  I16..4:  0.5%  1.0%  0.3%  P16..4:  3.0%  2.5%  1.4%  0.0%  0.0%    skip:91.4%
[libx264 @ 0x635fb1fd5640] mb B  I16..4:  0.1%  0.1%  0.0%  B16..8:  5.6%  1.7%  0.4%  direct: 0.1%  skip:91.9%  L0:49.0% L1:43.6% BI: 7.4%
[libx264 @ 0x635fb1fd5640] 8x8 transform intra:34.9% inter:22.1%
[libx264 @ 0x635fb1fd5640] coded y,uvDC,uvAC intra: 9.6% 14.1% 12.8% inter: 0.7% 1.9% 1.8%
[libx264 @ 0x635fb1fd5640] i16 v